# Introduction

Dataset reference : Bani, Z. (2023). An open‑access dataset of emergency department admissions at a large teaching hospital in Iran. Data Archiving and Networked Services (DANS). https://doi.org/10.17632/vhzyyktrz5.

The project at predicting hospitalisation from triage and/or spot the hospital stays that may be extended.

# Packages

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Data preprocessing

## Merging dataframes

In [2]:
df_admission = pd.read_csv('../Data/ED_admission.csv')
df_admission.columns

Index(['triage_code', 'PatientCode', 'DischargeFromED', 'Foreigners', 'gender',
       'age', 'marital_Status', 'ResidentDate_year', 'ResidentDate_month',
       'ResidentDate_day', 'ResidentDate_weekday', 'ResidentDate_hour',
       'DischargeDate_year', 'DischargeDate_month', 'DischargeDate_day',
       'DischargeDate_weekday', 'DischargeDate_hour', 'StatusOnDischarge',
       'KindRef', 'IsGlobal', 'AccompainerRelation_id', 'ResidentDay',
       'service_count_action', 'service_count_instrument', 'service_count_lab',
       'service_count_graphy'],
      dtype='str')

In [3]:
# Selecting useful columns from the administration data
cols_admission = [
    'triage_code',
    'PatientCode',       # Mandatory to group by patient during train/test split
    'DischargeFromED'    # Target
]

In [4]:
df_triage = pd.read_csv('../Data/ED_triage.csv')
df_triage.columns

Index(['triage_code', 'gender', 'age', 'admission_year', 'admission_month',
       'admission_day', 'admission_weekday', 'admission_hour', 'kindref',
       'ChiefComplaint', 'explainer_id', 'NeedFastExecute', 'CriticalStatus',
       'StuporStatus', 'PainGrade', 'MentalDistress', 'MaterialDistress',
       'Source', 'BlooddpressurSystol', 'BlooddpressurDiastol', 'PulseRate',
       'RespiratoryRate', 'Temperature', 'O2Saturation', 'AVPU', 'TriageGrade',
       'operational_patient', 'ref_specialist'],
      dtype='str')

In [5]:
# Selecting variables known during trial
cols_triage = [
    'triage_code',
    'gender', 'age', 'admission_weekday', 'admission_hour', # Demographic and temporal context
    'ChiefComplaint', 'kindref', 'Source', # Reason for and manner of arrival
    'TriageGrade', 'CriticalStatus', 'NeedFastExecute', 'StuporStatus', 'PainGrade', 'MentalDistress', 'MaterialDistress', 'AVPU', # Clinical scores and gravity at triage
    'BlooddpressurSystol', 'BlooddpressurDiastol', 'PulseRate', 'RespiratoryRate', 'Temperature', 'O2Saturation' # Vital Signs
]

In [6]:
df = pd.merge(df_admission[cols_admission], df_triage[cols_triage], on='triage_code', how='inner')

## Encoding variables

In [7]:
df.rename(columns={'BlooddpressurSystol' : 'BloodpressurSystol',
                   'BlooddpressurDiastol' : 'BloodpressurDiastol'}, 
                   inplace=True)

C:\Users\PC\AppData\Local\Temp\ipykernel_8100\1164776113.py:1: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={'BlooddpressurSystol' : 'BloodpressurSystol',


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 143516 entries, 0 to 143515
Data columns (total 24 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   triage_code          143516 non-null  int64  
 1   PatientCode          143516 non-null  int64  
 2   DischargeFromED      143516 non-null  int64  
 3   gender               143516 non-null  str    
 4   age                  143516 non-null  int64  
 5   admission_weekday    143516 non-null  int64  
 6   admission_hour       143516 non-null  int64  
 7   ChiefComplaint       143516 non-null  str    
 8   kindref              143516 non-null  int64  
 9   Source               56005 non-null   float64
 10  TriageGrade          143516 non-null  int64  
 11  CriticalStatus       133140 non-null  float64
 12  NeedFastExecute      143516 non-null  int64  
 13  StuporStatus         133140 non-null  float64
 14  PainGrade            133204 non-null  float64
 15  MentalDistress       133140 

In [9]:
df.isnull().sum()

triage_code                 0
PatientCode                 0
DischargeFromED             0
gender                      0
age                         0
admission_weekday           0
admission_hour              0
ChiefComplaint              0
kindref                     0
Source                  87511
TriageGrade                 0
CriticalStatus          10376
NeedFastExecute             0
StuporStatus            10376
PainGrade               10312
MentalDistress          10376
MaterialDistress        10376
AVPU                    90509
BloodpressurSystol     118848
BloodpressurDiastol    118546
PulseRate              108294
RespiratoryRate        110076
Temperature            143377
O2Saturation           105855
dtype: int64

In [ ]:
# Defining columns groups
num_vitals   = ['BloodpressurSystol', 'BloodpressurDiastol', 'PulseRate', 'RespiratoryRate', 'O2Saturation']
binary_cols  = ['NeedFastExecute', 'CriticalStatus', 'StuporStatus', 'MentalDistress', 'MaterialDistress']
nominal_cols = ['kindref', 'Source', 'AVPU']

# age, admission_weekday, admission_hour, TriageGrade, PainGrade are already numerical so no need to encode them

We have to handle missing values. For the vital signs, we will later fill them with the median and create a new column. I chosed this method not to choose an arbitrary value too far away from the physiological one, thus disturbing certain models. The missing-value indicators are created here, but the median is only computed on the training set after the train/test split to avoid data leakage. For the pain scale, I just filled with ``-1`` as the scale is 0-10 anyway.

In [ ]:
# Vital signs (new column only, NaN are kept: the median is filled after the split)
for col in num_vitals:
    df[f'{col}_missing'] = df[col].isna().astype(int)

# Filling pain scale
df['PainGrade'] = df['PainGrade'].fillna(-1)

# Binary indicators
df['triage_incomplete'] = df[binary_cols].isna().any(axis=1).astype(int)
df[binary_cols] = df[binary_cols].fillna(0).astype(int)

# Gender
df['gender'] = df['gender'].str.strip().str.lower().map({'male': 1, 'female': 0})
assert df['gender'].notna().all(), df['gender'].unique()
df['gender'] = df['gender'].astype(int)

In [12]:
df['ChiefComplaint'].nunique()

896

The ``ChiefComplaint`` values correspond to the International Classification of Diseases: ICD-10. Since 896 codes is obviously far too many, we’ll just look at the first letter. This will give us about 15-20 categories, which is much easier to interpret.

In [ ]:
# Categorical data
df['ChiefComplaint_category'] = df['ChiefComplaint'].str[0]

for col in nominal_cols:
    df[col] = df[col].astype('string').fillna('Unknown')

df = pd.get_dummies(
    df,
    columns=nominal_cols + ['ChiefComplaint_category'],
    prefix=['kindref', 'source', 'avpu', 'complaint'],
    drop_first=True,
    dtype=int,
)

``num_general`` has no missing values so I'm directling handling ``cat_cols``

In [ ]:
# Features and target
X = df.drop(columns=['triage_code', 'PatientCode', 'DischargeFromED', 'ChiefComplaint'])
y = df['DischargeFromED']
groups = df['PatientCode']

In [ ]:
# Train/test split grouped by patient
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[test_idx])

# Vital signs median computed on the training set only, applied to both sets
medians = X_train[num_vitals].median()
X_train[num_vitals] = X_train[num_vitals].fillna(medians)
X_test[num_vitals] = X_test[num_vitals].fillna(medians)